# TactiVision CV — Ball Detection + Interpolation (Colab GPU)

This notebook runs a **low-confidence ball-only detection** pass,
then applies interpolation to fill gaps.

**Goal:** Get ball coverage from 14% → 50%+

**Strategy:**
1. Detect ball at very low confidence (0.05) — catches faint/blurry balls
2. Filter false positives (static penalty spot, out-of-field)
3. Interpolate remaining gaps

**Setup:** Same as the tracker notebook — needs best.pt and match_5min.mp4 on Drive.

In [ ]:
# Cell 1: Install dependencies
!pip install -q ultralytics supervision scikit-learn

In [ ]:
# Cell 2: Mount Drive + set paths
from google.colab import drive
drive.mount('/content/drive')

import os

# EDIT THIS to match your Drive folder
DRIVE_FOLDER = '/content/drive/MyDrive/TactiScout-CV (1)'
MODEL_PATH = f'{DRIVE_FOLDER}/models/best.pt'
VIDEO_PATH = f'{DRIVE_FOLDER}/data/match_5min.mp4'
OUTPUT_FOLDER = f'{DRIVE_FOLDER}/output'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
assert os.path.exists(MODEL_PATH), f'Model not found at {MODEL_PATH}'
assert os.path.exists(VIDEO_PATH), f'Video not found at {VIDEO_PATH}'
print('✓ Files found')

In [ ]:
# Cell 3: Run ball-only detection at VERY low confidence
import supervision as sv
from ultralytics import YOLO
import numpy as np
import cv2
import json
import time

BALL_CLASS = 0

model = YOLO(MODEL_PATH)
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
fps = video_info.fps or 25
total_frames = video_info.total_frames

print(f'Video: {total_frames} frames, {fps} fps')
print(f'Running ball detection at conf=0.05...')
print(f'(This catches everything that even slightly looks like a ball)')

start = time.time()
ball_detections_raw = []

# Run detection ONLY for ball class at very low confidence
results_gen = model.predict(
    source=VIDEO_PATH,
    conf=0.05,           # Very low — catches faint balls
    classes=[BALL_CLASS], # Only ball class
    stream=True,
    verbose=False
)

for idx, results in enumerate(results_gen):
    boxes = results.boxes

    if boxes is None or len(boxes) == 0:
        continue

    confidences = boxes.conf.cpu().numpy()
    xyxy = boxes.xyxy.cpu().numpy()

    # Take the highest confidence ball in this frame
    best = confidences.argmax()
    x1, y1, x2, y2 = xyxy[best]

    ball_detections_raw.append({
        'frame': idx,
        'x': float((x1 + x2) / 2),
        'y': float((y1 + y2) / 2),
        'confidence': float(confidences[best]),
        'bbox_w': float(x2 - x1),
        'bbox_h': float(y2 - y1)
    })

    if idx % 1000 == 0:
        elapsed = time.time() - start
        speed = (idx + 1) / elapsed if elapsed > 0 else 0
        print(f'  Frame {idx}/{total_frames} | Speed: {speed:.0f} fps | '
              f'Ball found in {len(ball_detections_raw)} frames so far')

elapsed = time.time() - start
raw_coverage = len(ball_detections_raw) / total_frames * 100
print(f'\n=== Raw Ball Detection Done ===')
print(f'  Time: {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'  Ball detected in: {len(ball_detections_raw)}/{total_frames} frames ({raw_coverage:.1f}%)')

In [ ]:
# Cell 4: Filter false positives
print('=== Filtering False Positives ===')

# --- Filter 1: Remove static detections (penalty spot, center dot) ---
# If the ball stays at the same position for 20+ consecutive frames, it's static

sorted_dets = sorted(ball_detections_raw, key=lambda d: d['frame'])
static_threshold = 8.0   # pixels
static_min_frames = 15   # consecutive frames at same position

static_frames = set()
i = 0
while i < len(sorted_dets):
    cluster_x = sorted_dets[i]['x']
    cluster_y = sorted_dets[i]['y']
    j = i + 1
    while j < len(sorted_dets):
        dx = sorted_dets[j]['x'] - cluster_x
        dy = sorted_dets[j]['y'] - cluster_y
        if np.sqrt(dx**2 + dy**2) <= static_threshold:
            j += 1
        else:
            break
    if j - i >= static_min_frames:
        for k in range(i, j):
            static_frames.add(sorted_dets[k]['frame'])
    i = j if j > i + 1 else i + 1

after_static = [d for d in sorted_dets if d['frame'] not in static_frames]
print(f'  Static filter: {len(sorted_dets)} → {len(after_static)} '
      f'(removed {len(static_frames)} penalty spot frames)')

# --- Filter 2: Remove balls with unrealistic size ---
# The ball should be roughly 5-40px wide in broadcast footage
min_ball_size = 3
max_ball_size = 50

after_size = [d for d in after_static
              if min_ball_size <= d['bbox_w'] <= max_ball_size
              and min_ball_size <= d['bbox_h'] <= max_ball_size]
print(f'  Size filter: {len(after_static)} → {len(after_size)} '
      f'(removed {len(after_static) - len(after_size)} wrong-size detections)')

# --- Filter 3: Remove impossible speed jumps ---
# If ball jumps > 200px in 1 frame, one of the detections is wrong
# Keep the one that's more consistent with neighbors
max_jump = 200  # pixels per frame

after_speed = [after_size[0]] if after_size else []
for k in range(1, len(after_size)):
    prev = after_speed[-1] if after_speed else after_size[k]
    curr = after_size[k]
    frame_diff = curr['frame'] - prev['frame']
    if frame_diff == 0:
        continue
    dx = curr['x'] - prev['x']
    dy = curr['y'] - prev['y']
    speed = np.sqrt(dx**2 + dy**2) / frame_diff
    if speed <= max_jump:
        after_speed.append(curr)
    # If speed is too high, skip this detection (likely false positive)

print(f'  Speed filter: {len(after_size)} → {len(after_speed)} '
      f'(removed {len(after_size) - len(after_speed)} impossible jumps)')

ball_filtered = after_speed
filtered_coverage = len(ball_filtered) / total_frames * 100
print(f'\n  Filtered ball coverage: {len(ball_filtered)}/{total_frames} ({filtered_coverage:.1f}%)')

In [ ]:
# Cell 5: Interpolate gaps
print('=== Ball Interpolation ===')

max_gap = 75  # frames (~3 seconds at 25fps)
max_speed_interp = 80  # pixels per frame

# Sort by frame
ball_sorted = sorted(ball_filtered, key=lambda d: d['frame'])

ball_final = []
interpolated_count = 0

for i in range(len(ball_sorted)):
    # Add the real detection
    entry = ball_sorted[i].copy()
    entry['interpolated'] = False
    ball_final.append(entry)

    # Check if we can interpolate to next detection
    if i >= len(ball_sorted) - 1:
        continue

    current = ball_sorted[i]
    next_det = ball_sorted[i + 1]

    frame_gap = next_det['frame'] - current['frame']

    # Skip if consecutive or gap too large
    if frame_gap <= 1 or frame_gap > max_gap:
        continue

    # Check speed is reasonable
    dx = next_det['x'] - current['x']
    dy = next_det['y'] - current['y']
    distance = np.sqrt(dx**2 + dy**2)
    speed = distance / frame_gap

    if speed > max_speed_interp:
        continue

    # Linear interpolation
    for f in range(current['frame'] + 1, next_det['frame']):
        t = (f - current['frame']) / frame_gap
        ball_final.append({
            'frame': f,
            'x': float(current['x'] + t * dx),
            'y': float(current['y'] + t * dy),
            'confidence': 0.3 * (1 - min(t, 1-t)),  # Lower confidence in middle
            'interpolated': True
        })
        interpolated_count += 1

# Sort by frame
ball_final.sort(key=lambda d: d['frame'])

# Remove duplicates (keep highest confidence per frame)
from collections import defaultdict
frame_best = {}
for d in ball_final:
    f = d['frame']
    if f not in frame_best or d['confidence'] > frame_best[f]['confidence']:
        frame_best[f] = d
ball_final = sorted(frame_best.values(), key=lambda d: d['frame'])

final_coverage = len(ball_final) / total_frames * 100
real_count = sum(1 for d in ball_final if not d.get('interpolated', False))

print(f'  Real detections: {real_count}')
print(f'  Interpolated: {len(ball_final) - real_count}')
print(f'  Total frames with ball: {len(ball_final)}/{total_frames}')
print(f'\n  FINAL BALL COVERAGE: {final_coverage:.1f}% (was 14%)')

In [ ]:
# Cell 6: Save results
output_path = f'{OUTPUT_FOLDER}/ball_final.json'

# Clean up for JSON serialization (remove bbox_w/h from output)
ball_output = []
for d in ball_final:
    ball_output.append({
        'frame': d['frame'],
        'x': d['x'],
        'y': d['y'],
        'confidence': d['confidence'],
        'interpolated': d.get('interpolated', False)
    })

with open(output_path, 'w') as f:
    json.dump(ball_output, f)

print(f'✓ Saved: {output_path}')
print(f'\n=== Summary ===')
print(f'  Raw detection (conf=0.05): {len(ball_detections_raw)}/{total_frames} ({len(ball_detections_raw)/total_frames*100:.1f}%)')
print(f'  After filtering:           {len(ball_filtered)}/{total_frames} ({len(ball_filtered)/total_frames*100:.1f}%)')
print(f'  After interpolation:       {len(ball_final)}/{total_frames} ({final_coverage:.1f}%)')
print(f'  Improvement: 14% → {final_coverage:.1f}%')